# ⚡ Kaggle Notebook 1 — Complete Training
## Quantum-Enhanced Deep Learning for Night-Time LPR

### Kaggle-specific features:
- **Dual T4 GPU** — DataParallel for classical layers (CNN, LSTM)
- **HuggingFace Hub** — all checkpoints pulled/pushed to HF (survives session restart)
- **Auto-resume** — restarts from the latest epoch on HF automatically
- **No Google Drive** — zero Drive dependency

---
### Before running:
1. Enable **GPU T4 x2** accelerator in Kaggle settings
2. Add **Kaggle Secrets**:
   - `HF_TOKEN_1` = `ADD_YOUR_HF_TOKEN_1_IN_KAGGLE_SECRETS`
   - `HF_TOKEN_2` = `ADD_YOUR_HF_TOKEN_2_IN_KAGGLE_SECRETS`
3. Run **00_Setup_HuggingFace.ipynb** on Colab first (one time only)

## Cell 1 — Install & GPU Check

In [1]:
print("he")

he


In [2]:
!pip install pennylane pennylane-lightning huggingface_hub editdistance -q

import os, json, time, zipfile
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pennylane as qml
from tqdm import tqdm
import editdistance
from huggingface_hub import HfApi, login, whoami, hf_hub_download

# ── GPU Setup ──────────────────────────────────────────────
n_gpus = torch.cuda.device_count()
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

print(f'🖥️  Available GPUs: {n_gpus}')
for i in range(n_gpus):
    props = torch.cuda.get_device_properties(i)
    print(f'   GPU {i}: {props.name}  ({props.total_memory/1e9:.1f} GB)')
print(f'\n✅ Primary device: {device}')
print(f'✅ PennyLane: {qml.__version__}')

# Use both GPUs for classical layers; quantum stays on GPU 0
USE_MULTI_GPU = (n_gpus >= 2)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 57.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 935.6/935.6 kB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.9/167.9 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 66.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 61.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 98.4 MB/s eta 0:00:00:00:0100:01
🖥️  Available GPUs: 2
   GPU 0: Tesla T4  (15.6 GB)
   GPU 1: Tesla T4  (15.6 GB)

✅ Primary device: cuda:0
✅ PennyLane: 0.44.1


## Cell 2 — HuggingFace Authentication (from Kaggle Secrets)

In [3]:
from kaggle_secrets import UserSecretsClient

HF_USERNAME     = 'Shanmuk4622'
HF_DATASET_REPO = f'{HF_USERNAME}/quantum-lpr-dataset'
HF_MODEL_REPO   = f'{HF_USERNAME}/quantum-lpr-checkpoints'

# Pull secrets from Kaggle
secrets = UserSecretsClient()
HF_TOKENS = []
for key in ['HF_TOKEN_1', 'HF_TOKEN_2']:
    try:
        HF_TOKENS.append(secrets.get_secret(key))
    except Exception:
        pass

if not HF_TOKENS:
    # Fallback: hardcoded (for testing; prefer Secrets)
    HF_TOKENS = [
        'ADD_YOUR_HF_TOKEN_1_IN_KAGGLE_SECRETS',
        'ADD_YOUR_HF_TOKEN_2_IN_KAGGLE_SECRETS',
    ]
    print('⚠️  Kaggle Secrets not found — using hardcoded tokens.')

# Authenticate with first working token
active_token = None
api = None
for tok in HF_TOKENS:
    try:
        login(token=tok, add_to_git_credential=False)
        info = whoami(token=tok)
        active_token = tok
        api = HfApi(token=tok)
        print(f'✅ HF authenticated as: {info["name"]}')
        break
    except Exception as e:
        print(f'⚠️  Token failed: {e}')

if not active_token:
    raise RuntimeError('All HF tokens failed. Check Kaggle Secrets.')

print(f'  Dataset repo  : {HF_DATASET_REPO}')
print(f'  Checkpoint repo: {HF_MODEL_REPO}')

⚠️  Kaggle Secrets not found — using hardcoded tokens.
✅ HF authenticated as: Shanmuk4622
  Dataset repo  : Shanmuk4622/quantum-lpr-dataset
  Checkpoint repo: Shanmuk4622/quantum-lpr-checkpoints


## Cell 3 — Download Dataset from HuggingFace

In [4]:
import os
import requests
import zipfile

WORK_DIR    = '/kaggle/working'
DATA_DIR    = WORK_DIR + '/lpr_data'
EXTRACT_DIR = DATA_DIR + '/images'
os.makedirs(DATA_DIR, exist_ok=True)



CSV_LOCAL   = DATA_DIR + '/2_train_hr_images.csv'
ZIP_LOCAL   = DATA_DIR + '/train.zip'


# ── Path definitions (add back what was dropped) ─────────
import os, json

WORK_DIR    = '/kaggle/working'
DATA_DIR    = WORK_DIR + '/lpr_data'
EXTRACT_DIR = DATA_DIR + '/images'
CKPT_DIR    = WORK_DIR + '/checkpoints'       # ← this was missing
CSV_LOCAL   = DATA_DIR + '/2_train_hr_images.csv'
ZIP_LOCAL   = DATA_DIR + '/train.zip'

os.makedirs(DATA_DIR,   exist_ok=True)
os.makedirs(CKPT_DIR,   exist_ok=True)
os.makedirs(EXTRACT_DIR, exist_ok=True)

print('✅ Paths defined:')
print(f'  DATA_DIR    : {DATA_DIR}')
print(f'  EXTRACT_DIR : {EXTRACT_DIR}')
print(f'  CKPT_DIR    : {CKPT_DIR}')
print(f'  CSV_LOCAL   : {CSV_LOCAL}')


print('🧹 Cleaning up any broken files...')
if os.path.exists(ZIP_LOCAL) and os.path.getsize(ZIP_LOCAL) < 700_000_000:
    os.remove(ZIP_LOCAL)
    print("  ...deleted broken zip.")

headers = {"Authorization": f"Bearer {active_token}"}

# 1. Download CSV
if not os.path.exists(CSV_LOCAL):
    print('\n⬇️  Downloading CSV...')
    csv_url = f"https://huggingface.co/datasets/{HF_DATASET_REPO}/resolve/main/data/2_train_hr_images.csv"
    r_csv = requests.get(csv_url, headers=headers)
    with open(CSV_LOCAL, 'wb') as f:
        f.write(r_csv.content)
    print('  ✅ CSV complete.')

# 2. Download ZIP (Streaming chunk by chunk to avoid memory locks)
if not os.path.exists(ZIP_LOCAL):
    print('\n⬇️  Downloading ZIP manually (Bulletproof streaming)...')
    zip_url = f"https://huggingface.co/datasets/{HF_DATASET_REPO}/resolve/main/data/wYe7pBJ7-train.zip"
    
    with requests.get(zip_url, headers=headers, stream=True) as r:
        r.raise_for_status()
        total_size = int(r.headers.get('content-length', 0))
        downloaded = 0
        
        with open(ZIP_LOCAL, 'wb') as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):  # 1 MB chunks
                if chunk:
                    f.write(chunk)
                    downloaded += len(chunk)
                    # Print progress every 50 MB to not spam the exact output
                    if downloaded % (50 * 1024 * 1024) == 0:
                        print(f"      Downloaded {downloaded / 1024 / 1024:.0f} MB / {total_size / 1024 / 1024:.0f} MB")
                        
    print("  ✅ ZIP download complete.")

# 3. Extract the ZIP
if not os.path.exists(EXTRACT_DIR) or len(os.listdir(EXTRACT_DIR)) < 5:
    print('\n📂 Extracting ZIP...')
    with zipfile.ZipFile(ZIP_LOCAL, 'r') as z:
        z.extractall(EXTRACT_DIR)
    print(f'✅ Extracted to {EXTRACT_DIR}')
else:
    print(f'\n✅ Already extracted ({len(os.listdir(EXTRACT_DIR))} items)')


✅ Paths defined:
  DATA_DIR    : /kaggle/working/lpr_data
  EXTRACT_DIR : /kaggle/working/lpr_data/images
  CKPT_DIR    : /kaggle/working/checkpoints
  CSV_LOCAL   : /kaggle/working/lpr_data/2_train_hr_images.csv
🧹 Cleaning up any broken files...

⬇️  Downloading CSV...
  ✅ CSV complete.

⬇️  Downloading ZIP manually (Bulletproof streaming)...
      Downloaded 50 MB / 686 MB
      Downloaded 100 MB / 686 MB
      Downloaded 150 MB / 686 MB
      Downloaded 200 MB / 686 MB
      Downloaded 250 MB / 686 MB
      Downloaded 300 MB / 686 MB
      Downloaded 350 MB / 686 MB
      Downloaded 400 MB / 686 MB
      Downloaded 450 MB / 686 MB
      Downloaded 500 MB / 686 MB
      Downloaded 550 MB / 686 MB
      Downloaded 600 MB / 686 MB
      Downloaded 650 MB / 686 MB
  ✅ ZIP download complete.

📂 Extracting ZIP...
✅ Extracted to /kaggle/working/lpr_data/images


## Cell 4 — Config & Hyperparameters

In [5]:
# ─────────────────────────────────────────────────────────────
# Hyperparameters — Kaggle-optimized
# ─────────────────────────────────────────────────────────────
BATCH_SIZE   = 32 if USE_MULTI_GPU else 16  # Larger batch w/ dual GPU
LR_INIT      = 0.001
TOTAL_EPOCHS = 100
PATIENCE     = 12    # Slightly more patient on faster hardware
TRAIN_RATIO  = 0.70
VAL_RATIO    = 0.15
SEED         = 42

# HF paths for checkpoints
HF_Q_LATEST  = 'quantum/latest.pth'
HF_Q_BEST    = 'quantum/best.pth'
HF_Q_HIST    = 'quantum/history.json'
HF_C_LATEST  = 'classical/latest.pth'
HF_C_BEST    = 'classical/best.pth'
HF_C_HIST    = 'classical/history.json'

CHARS    = '0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ'
CHAR2IDX = {c: i+1 for i, c in enumerate(CHARS)}
IDX2CHAR = {i+1: c for i, c in enumerate(CHARS)}
N_QUBITS = 8
N_LAYERS = 2

print(f'✅ Config:')
print(f'   Batch size    : {BATCH_SIZE} ({"dual GPU" if USE_MULTI_GPU else "single GPU"})')
print(f'   Total epochs  : {TOTAL_EPOCHS}')
print(f'   LR init       : {LR_INIT}')
print(f'   Early stopping: patience={PATIENCE}')

✅ Config:
   Batch size    : 32 (dual GPU)
   Total epochs  : 100
   LR init       : 0.001
   Early stopping: patience=12


## Cell 5 — Dataset

In [6]:
import os

WORK_DIR = '/kaggle/working'
CKPT_DIR = WORK_DIR + '/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)

print(f"✅ CKPT_DIR fixed: {CKPT_DIR}")


class LPRDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None, mode='train'):
        self.df        = pd.read_csv(csv_file)
        self.root_dir  = root_dir
        self.transform = transform
        self.mode      = mode

    def __len__(self): return len(self.df)

    def encode_text(self, text):
        return [CHAR2IDX[c] for c in str(text).upper() if c in CHAR2IDX]

    def make_night(self, img):
        if torch.rand(1) < 0.5:
            g   = np.random.uniform(2.0, 3.5)
            img = torch.pow(img, g)
            img = torch.clamp(img + torch.randn_like(img) * 0.05, 0, 1)
        return img

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        p    = str(row['path'])
        full = p if p.startswith('/') else os.path.join(self.root_dir, p)
        if not os.path.exists(full):
            # Try under EXTRACT_DIR/train
            alt = os.path.join(EXTRACT_DIR, 'train', os.path.basename(p))
            full = alt if os.path.exists(alt) else full
        try:   img = Image.open(full).convert('RGB')
        except: img = Image.new('RGB', (256, 64))
        if self.transform: img = self.transform(img)
        if self.mode == 'train': img = self.make_night(img)
        enc = torch.tensor(self.encode_text(str(row['label'])), dtype=torch.long)
        return img, enc, torch.tensor(len(enc), dtype=torch.long)


def collate_fn(batch):
    imgs, labs, lens = zip(*batch)
    return torch.stack(imgs, 0), torch.cat(labs), torch.stack(lens, 0)


transform = transforms.Compose([transforms.Resize((64, 256)), transforms.ToTensor()])

torch.manual_seed(SEED)
full_ds = LPRDataset(CSV_LOCAL, EXTRACT_DIR, transform=transform, mode='train')
N       = len(full_ds)
n_train = int(N * TRAIN_RATIO)
n_val   = int(N * VAL_RATIO)
n_test  = N - n_train - n_val

train_set, val_set, test_set = random_split(full_ds, [n_train, n_val, n_test])

eval_ds = LPRDataset(CSV_LOCAL, EXTRACT_DIR, transform=transform, mode='eval')
val_set.dataset  = eval_ds
test_set.dataset = eval_ds

# More workers on Kaggle (4 CPUs available)
WORKERS = 4
train_loader = DataLoader(train_set, BATCH_SIZE, shuffle=True,  collate_fn=collate_fn, num_workers=WORKERS, pin_memory=True)
val_loader   = DataLoader(val_set,   BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=WORKERS, pin_memory=True)
test_loader  = DataLoader(test_set,  BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=WORKERS, pin_memory=True)

# Save test indices to HF for reproducibility (upload once)
test_idx_local = os.path.join(CKPT_DIR, 'test_indices.json')
with open(test_idx_local, 'w') as f:
    json.dump(list(test_set.indices), f)
try:
    api.upload_file(path_or_fileobj=test_idx_local, path_in_repo='meta/test_indices.json',
                    repo_id=HF_MODEL_REPO, repo_type='model', token=active_token)
except Exception:
    pass  # Already there from setup

print(f'✅ Dataset: Train={n_train}  Val={n_val}  Test={n_test}')

✅ CKPT_DIR fixed: /kaggle/working/checkpoints


No files have been modified since last commit. Skipping to prevent empty commit.


✅ Dataset: Train=70000  Val=15000  Test=15000


## Cell 6 — Model Definitions
### Dual-GPU Strategy:
- `ZeroDCE + CNN + LSTM + Classifier` → wrapped in `DataParallel` (uses both GPUs)
- `QuantumLayer` → stays on `cuda:0` only (PennyLane QNode limitation)

In [7]:
# ── ZeroDCE ───────────────────────────────────────────────
class ZeroDCE_Light(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3,16,3,1,1), nn.ReLU(),
            nn.Conv2d(16,16,3,1,1), nn.ReLU(),
            nn.Conv2d(16,24,3,1,1), nn.Tanh())
    def forward(self, x):
        A, e = self.conv(x), x
        for i in range(8): e = e + A[:,3*i:3*(i+1)] * (torch.pow(e,2) - e)
        return e

# ── Quantum Circuit ──────────────────────────────────────
# Always on cuda:0 — do NOT DataParallel this
dev_qml = qml.device('default.qubit', wires=N_QUBITS)

@qml.qnode(dev_qml, interface='torch')
def quantum_circuit(inputs, weights):
    qml.templates.AngleEmbedding(inputs, wires=range(N_QUBITS))
    qml.templates.StronglyEntanglingLayers(weights, wires=range(N_QUBITS))
    return [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS)]

class QuantumLayer(nn.Module):
    def __init__(self):
        super().__init__()
        self.q_layer = qml.qnn.TorchLayer(quantum_circuit,
                                           {'weights': (N_LAYERS, N_QUBITS, 3)})
    def forward(self, x):
        # x is always on cuda:0 (QNode requirement)
        return self.q_layer(x)

# ── Hybrid Quantum Model ──────────────────────────────────
class HybridLPRNet_8Q(nn.Module):
    """
    Dual-GPU aware HybridLPRNet.
    Classical path runs on DataParallel (both GPUs).
    Quantum circuit always runs on cuda:0.
    """
    def __init__(self, num_classes=37):
        super().__init__()
        self.enhancer   = ZeroDCE_Light()
        self.features   = nn.Sequential(
            nn.Conv2d(3,64,3,1,1), nn.MaxPool2d(2), nn.ReLU(),
            nn.Conv2d(64,128,3,1,1), nn.MaxPool2d(2), nn.ReLU(),
            nn.Conv2d(128,N_QUBITS,1,1))
        self.quantum    = QuantumLayer()       # GPU 0 only
        self.rnn        = nn.LSTM(N_QUBITS, 128, bidirectional=True, batch_first=True)
        self.classifier = nn.Linear(256, num_classes)

    def forward(self, x):
        # x may come from any GPU when using DataParallel
        # Move to cuda:0 before quantum layer
        x   = self.enhancer(x)
        x   = self.features(x)
        b,c,h,w = x.size()
        x_seq = x.mean(dim=2).permute(0,2,1)   # [B, W, 8]
        # Quantum circuit: must be on cuda:0
        x0    = x_seq.to('cuda:0').reshape(-1, N_QUBITS)
        q_out = self.quantum(x0).reshape(b, w, N_QUBITS)
        # Move back to current device for LSTM
        q_out = q_out.to(x_seq.device)
        rnn_out, _ = self.rnn(q_out)
        return self.classifier(rnn_out).permute(1,0,2)  # [T, B, C]

    def get_pre_quantum(self, x):
        with torch.no_grad():
            x = self.enhancer(x)
            x = self.features(x)
            return x.mean(dim=2).permute(0,2,1)

    def get_post_quantum(self, x):
        with torch.no_grad():
            pre = self.get_pre_quantum(x)
            b,w,_ = pre.shape
            return self.quantum(pre.to('cuda:0').reshape(-1,N_QUBITS)).reshape(b,w,N_QUBITS)

# ── Classical Baseline ──────────────────────────────────
class ClassicalLPRNet(nn.Module):
    def __init__(self, num_classes=37):
        super().__init__()
        self.enhancer   = ZeroDCE_Light()
        self.features   = nn.Sequential(
            nn.Conv2d(3,64,3,1,1), nn.MaxPool2d(2), nn.ReLU(),
            nn.Conv2d(64,128,3,1,1), nn.MaxPool2d(2), nn.ReLU(),
            nn.Conv2d(128,N_QUBITS,1,1))
        self.classical  = nn.Sequential(
            nn.Linear(N_QUBITS,16), nn.Tanh(), nn.Linear(16,N_QUBITS))
        self.rnn        = nn.LSTM(N_QUBITS, 128, bidirectional=True, batch_first=True)
        self.classifier = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.enhancer(x)
        x = self.features(x)
        b,c,h,w = x.size()
        x = self.classical(x.mean(dim=2).permute(0,2,1))
        return self.classifier(self.rnn(x)[0]).permute(1,0,2)

# ── Instantiate & wrap for multi-GPU ────────────────────
q_model = HybridLPRNet_8Q(37).to(device)
c_model = ClassicalLPRNet(37).to(device)

# Wrap EVERYTHING including quantum in DataParallel
# The forward() method internally routes quantum to cuda:0
if USE_MULTI_GPU:
    q_model = nn.DataParallel(q_model)
    c_model = nn.DataParallel(c_model)
    print(f'✅ DataParallel enabled across {n_gpus} GPUs')

def count_params(model):
    m = model.module if isinstance(model, nn.DataParallel) else model
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

print(f'⚡ Quantum  params: {count_params(q_model):,}')
print(f'🔷 Classical params: {count_params(c_model):,}')

# ── Cell 6 FIX: Re-instantiate models WITHOUT DataParallel ──────────
# Reason: model outputs [T, B, C] (sequence-first), which DataParallel
# incorrectly concatenates along T dim instead of B dim.
# Quantum circuit already forces cuda:0 anyway — DP gives no real benefit.

import importlib
import torch, torch.nn as nn
import pennylane as qml

N_QUBITS = 8; N_LAYERS = 2
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# Re-define quantum circuit fresh (avoids any stale DP state)
dev_qml = qml.device('default.qubit', wires=N_QUBITS)

@qml.qnode(dev_qml, interface='torch')
def quantum_circuit(inputs, weights):
    qml.templates.AngleEmbedding(inputs, wires=range(N_QUBITS))
    qml.templates.StronglyEntanglingLayers(weights, wires=range(N_QUBITS))
    return [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS)]

class QuantumLayer(nn.Module):
    def __init__(self): super().__init__(); self.q_layer = qml.qnn.TorchLayer(quantum_circuit, {'weights': (N_LAYERS, N_QUBITS, 3)})
    def forward(self, x): return self.q_layer(x)

# Re-instantiate — plain .to(device), NO DataParallel
q_model = HybridLPRNet_8Q(37).to(device)
c_model = ClassicalLPRNet(37).to(device)

def count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)
print(f'⚡ Quantum  params: {count_params(q_model):,}')
print(f'🔷 Classical params: {count_params(c_model):,}')
print(f'✅ Using single GPU: {torch.cuda.get_device_name(0)}')
print(f'   (Dual T4 VRAM available but quantum circuit forces single-device execution)')


✅ DataParallel enabled across 2 GPUs
⚡ Quantum  params: 233,797
🔷 Classical params: 234,029
⚡ Quantum  params: 233,797
🔷 Classical params: 234,029
✅ Using single GPU: Tesla T4
   (Dual T4 VRAM available but quantum circuit forces single-device execution)


## Cell 7 — HuggingFace Checkpoint Helpers

In [8]:
def get_raw_model(model):
    """Unwrap DataParallel to get the base model."""
    return model.module if isinstance(model, nn.DataParallel) else model


def push_checkpoint_to_hf(model, optimizer, epoch, val_cer, hf_path,
                           history=None, hist_hf_path=None):
    """Save checkpoint locally then push to HF Hub."""
    local_path = os.path.join(CKPT_DIR, os.path.basename(hf_path))
    torch.save({
        'epoch': epoch,
        'model_state_dict': get_raw_model(model).state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'val_cer': val_cer,
    }, local_path)
    try:
        api.upload_file(
            path_or_fileobj=local_path,
            path_in_repo=hf_path,
            repo_id=HF_MODEL_REPO,
            repo_type='model',
            token=active_token,
            commit_message=f'Epoch {epoch+1} — val_CER={val_cer:.4f}'
        )
    except Exception as e:
        print(f'  ⚠️  HF push failed (local copy kept): {e}')

    # Also push history
    if history and hist_hf_path:
        hist_local = os.path.join(CKPT_DIR, os.path.basename(hist_hf_path))
        with open(hist_local, 'w') as f:
            json.dump(history, f)
        try:
            api.upload_file(
                path_or_fileobj=hist_local,
                path_in_repo=hist_hf_path,
                repo_id=HF_MODEL_REPO,
                repo_type='model',
                token=active_token,
                commit_message=f'History update epoch {epoch+1}'
            )
        except Exception:
            pass


def pull_checkpoint_from_hf(model, optimizer, hf_latest, hf_hist):
    """Download latest checkpoint from HF and load it. Returns start_epoch."""
    # Try to download latest.pth
    local_path = os.path.join(CKPT_DIR, 'latest.pth')
    try:
        cached = hf_hub_download(
            repo_id=HF_MODEL_REPO, filename=hf_latest,
            repo_type='model', token=active_token,
            local_dir=CKPT_DIR, local_dir_use_symlinks=False,
            force_download=True  # Always get latest from HF
        )
        ckpt = torch.load(cached, map_location=device)
        get_raw_model(model).load_state_dict(ckpt['model_state_dict'])
        if optimizer:
            optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        start = ckpt.get('epoch', -1) + 1
        cer   = ckpt.get('val_cer', '?')
        print(f'  ✅ Resumed from HF epoch {start} (val_CER={cer})')
        return start
    except Exception as e:
        print(f'  ℹ️  No checkpoint found on HF: {e}')
        return 0


def pull_history_from_hf(hf_hist_path):
    """Download history JSON from HF."""
    try:
        cached = hf_hub_download(
            repo_id=HF_MODEL_REPO, filename=hf_hist_path,
            repo_type='model', token=active_token,
            local_dir=CKPT_DIR, local_dir_use_symlinks=False,
            force_download=True
        )
        with open(cached) as f:
            hist = json.load(f)
        print(f'  ✅ Loaded history: {len(hist["val_cer"])} epochs recorded')
        return hist
    except Exception:
        return {'train_loss':[], 'val_loss':[], 'val_cer':[], 'val_wer':[]}


# CTC utilities
def ctc_greedy_decode(log_probs):
    indices = torch.argmax(log_probs, dim=2)
    batch = []
    for b in range(indices.size(1)):
        seq, chars, prev = indices[:,b].tolist(), [], -1
        for i in seq:
            if i != 0 and i != prev: chars.append(IDX2CHAR.get(i,''))
            prev = i
        batch.append(''.join(chars))
    return batch

def decode_targets(targets, lengths):
    strings, offset = [], 0
    for l in lengths.tolist():
        strings.append(''.join(IDX2CHAR.get(i,'') for i in targets[offset:offset+l].tolist()))
        offset += l
    return strings

def compute_cer_wer(model, loader):
    model.eval()
    total_cer, total_wer, count = 0.0, 0, 0
    with torch.no_grad():
        for imgs, targets, lengths in loader:
            preds   = model(imgs.to(device)).cpu()
            decoded = ctc_greedy_decode(preds)
            actuals = decode_targets(targets, lengths)
            for p, t in zip(decoded, actuals):
                if len(t) > 0:
                    total_cer += editdistance.eval(p, t) / len(t)
                total_wer += (0 if p == t else 1)
                count += 1
    return (total_cer/count if count else 1.0, total_wer/count if count else 1.0)

print('✅ Checkpoint helpers & utilities ready.')

✅ Checkpoint helpers & utilities ready.


## Cell 8 — Training Loop

In [9]:
def train_model(model, model_name, hf_latest, hf_best, hf_hist, resume=True):
    """
    Full training loop with:
    - Auto-resume from HuggingFace
    - Push checkpoint to HF every epoch
    - DataParallel-aware
    - Cosine LR + early stopping
    """
    optimizer = optim.Adam(model.parameters(), lr=LR_INIT)
    criterion = nn.CTCLoss(blank=0, zero_infinity=True)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=TOTAL_EPOCHS)

    start_epoch = 0
    if resume:
        print(f'[ {model_name} ] Checking HF for checkpoint...')
        start_epoch = pull_checkpoint_from_hf(model, optimizer, hf_latest, hf_hist)
        # Fast-forward scheduler to match epochs already trained
        for _ in range(start_epoch):
            scheduler.step()

    history  = pull_history_from_hf(hf_hist) if resume else \
               {'train_loss':[], 'val_loss':[], 'val_cer':[], 'val_wer':[]}
    best_cer = min(history['val_cer']) if history['val_cer'] else 1.0
    no_improve = 0

    print(f'\n🚀 [{model_name}] Training — Epoch {start_epoch+1}/{TOTAL_EPOCHS}')
    if start_epoch > 0:
        print(f'   (Resuming — best CER so far: {best_cer:.4f})')
    print('=' * 70)

    for epoch in range(start_epoch, TOTAL_EPOCHS):
        # ── Train ────────────────────────────────────────
        model.train()
        epoch_loss = 0.0
        t_epoch    = time.time()
        pbar = tqdm(train_loader, desc=f'Ep {epoch+1:3d}/{TOTAL_EPOCHS} [{model_name}]')
        for imgs, targets, lengths in pbar:
            imgs    = imgs.to(device)
            preds   = model(imgs)           # [T, B, C]
            T, B    = preds.size(0), imgs.size(0)
            il      = torch.full((B,), T, dtype=torch.long, device=device)
            loss    = criterion(preds.log_softmax(2), targets, il, lengths)
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
            epoch_loss += loss.item()
            pbar.set_postfix(loss=f'{loss.item():.4f}')

        optimizer.step()  # Required BEFORE scheduler.step() in PyTorch
        scheduler.step()
        avg_train_loss = epoch_loss / len(train_loader)
        epoch_time     = (time.time() - t_epoch) / 60

        # ── Validate ─────────────────────────────────────
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for imgs, targets, lengths in val_loader:
                imgs = imgs.to(device)
                preds= model(imgs)
                T, B = preds.size(0), imgs.size(0)
                il   = torch.full((B,), T, dtype=torch.long, device=device)
                val_loss += criterion(preds.log_softmax(2), targets, il, lengths).item()
        avg_val_loss = val_loss / len(val_loader)
        val_cer, val_wer = compute_cer_wer(model, val_loader)

        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['val_cer'].append(val_cer)
        history['val_wer'].append(val_wer)

        print(f'  Ep{epoch+1:3d} | Train={avg_train_loss:.4f} | Val={avg_val_loss:.4f} | '
              f'CER={val_cer:.4f} | WER={val_wer:.4f} | {epoch_time:.1f}min')

        # ── Push latest to HF every epoch ────────────────
        push_checkpoint_to_hf(model, optimizer, epoch, val_cer,
                               hf_latest, history, hf_hist)

        # ── Best model ───────────────────────────────────
        if val_cer < best_cer:
            best_cer   = val_cer
            no_improve = 0
            push_checkpoint_to_hf(model, optimizer, epoch, val_cer, hf_best)
            print(f'  🏆 New best! CER={best_cer:.4f} → pushed {hf_best}')
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                print(f'  ⏹  Early stopping (no improvement for {PATIENCE} epochs)')
                break

    print(f'\n✅ [{model_name}] Done. Best val CER: {best_cer:.4f}')
    return history

print('✅ Training loop ready.')

✅ Training loop ready.


In [10]:
# ── TQDM Fix: stop double-printing in Kaggle notebooks ─────────
from tqdm.auto import tqdm as tqdm_auto

def train_model(model, model_name, hf_latest, hf_best, hf_hist, resume=True):
    optimizer = optim.Adam(model.parameters(), lr=LR_INIT)
    criterion = nn.CTCLoss(blank=0, zero_infinity=True)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=TOTAL_EPOCHS)

    start_epoch = 0
    if resume:
        print(f'[ {model_name} ] Checking HF for checkpoint...')
        start_epoch = pull_checkpoint_from_hf(model, optimizer, hf_latest, hf_hist)
        for _ in range(start_epoch):
            scheduler.step()

    history  = pull_history_from_hf(hf_hist) if resume else \
               {'train_loss':[], 'val_loss':[], 'val_cer':[], 'val_wer':[]}
    best_cer = min(history['val_cer']) if history['val_cer'] else 1.0
    no_improve = 0

    print(f'\n🚀 [{model_name}] Training — Epoch {start_epoch+1}/{TOTAL_EPOCHS}')
    if start_epoch > 0:
        print(f'   (Resuming — best CER so far: {best_cer:.4f})')
    print('=' * 70)

    for epoch in range(start_epoch, TOTAL_EPOCHS):
        model.train()
        epoch_loss = 0.0
        t_epoch    = time.time()

        # ✅ FIX: tqdm.auto handles Kaggle/Jupyter correctly
        pbar = tqdm_auto(train_loader,
                         desc=f'Ep {epoch+1:3d}/{TOTAL_EPOCHS} [{model_name}]',
                         leave=True)

        for imgs, targets, lengths in pbar:
            imgs    = imgs.to(device)
            preds   = model(imgs)
            T, B    = preds.size(0), preds.size(1)   # use preds.size(1), not imgs.size(0)
            il      = torch.full((B,), T, dtype=torch.long, device=device)
            loss    = criterion(preds.log_softmax(2), targets, il, lengths)
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
            epoch_loss += loss.item()

            # ✅ FIX: refresh=False stops the double-print
            pbar.set_postfix(loss=f'{loss.item():.4f}', refresh=False)

        optimizer.step()
        scheduler.step()
        avg_train_loss = epoch_loss / len(train_loader)
        epoch_time     = (time.time() - t_epoch) / 60

        # ── Validate ─────────────────────────────────────
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for imgs, targets, lengths in val_loader:
                imgs  = imgs.to(device)
                preds = model(imgs)
                T, B  = preds.size(0), preds.size(1)
                il    = torch.full((B,), T, dtype=torch.long, device=device)
                val_loss += criterion(preds.log_softmax(2), targets, il, lengths).item()
        avg_val_loss = val_loss / len(val_loader)
        val_cer, val_wer = compute_cer_wer(model, val_loader)

        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['val_cer'].append(val_cer)
        history['val_wer'].append(val_wer)

        print(f'  Ep{epoch+1:3d} | Train={avg_train_loss:.4f} | Val={avg_val_loss:.4f} | '
              f'CER={val_cer:.4f} | WER={val_wer:.4f} | {epoch_time:.1f}min')

        push_checkpoint_to_hf(model, optimizer, epoch, val_cer, hf_latest, history, hf_hist)

        if val_cer < best_cer:
            best_cer = val_cer; no_improve = 0
            push_checkpoint_to_hf(model, optimizer, epoch, val_cer, hf_best)
            print(f'  🏆 New best! CER={best_cer:.4f} → {hf_best}')
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                print(f'  ⏹  Early stopping ({PATIENCE} epochs no improvement)')
                break

    print(f'\n✅ [{model_name}] Done. Best CER: {best_cer:.4f}')
    return history

print('✅ Training loop redefined (double-print fix applied)')


✅ Training loop redefined (double-print fix applied)


## Cell 9 — 🚀 Train Quantum Model (auto-resumes from HF)

In [11]:
##### print('⚡ Starting QUANTUM model training...')
quantum_history = train_model(
    model      = q_model,
    model_name = 'Quantum',
    hf_latest  = HF_Q_LATEST,
    hf_best    = HF_Q_BEST,
    hf_hist    = HF_Q_HIST,
    resume     = True   # ← Pulls from HF automatically
)

[ Quantum ] Checking HF for checkpoint...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


quantum/latest.pth:   0%|          | 0.00/2.83M [00:00<?, ?B/s]

  ✅ Resumed from HF epoch 100 (val_CER=0.01585714285714265)


/tmp/ipykernel_55/1760243901.py:14: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


history.json: 0.00B [00:00, ?B/s]

  ✅ Loaded history: 97 epochs recorded

🚀 [Quantum] Training — Epoch 101/100
   (Resuming — best CER so far: 0.0156)

✅ [Quantum] Done. Best CER: 0.0156


## Cell 10 — 🔷 Train Classical Baseline

In [12]:
print("hi")

hi


In [ ]:
# 🔷 Retrain Classical Baseline (fresh start)

print("🔷 Starting CLASSICAL model training from scratch (no resume)...")

classical_history_fresh = train_model(
    model      = c_model,
    model_name = 'Classical',
    hf_latest  = HF_C_LATEST,
    hf_best    = HF_C_BEST,
    hf_hist    = HF_C_HIST,
    resume     = False   # Forces start from epoch 0, ignores any leftover local files
)

print("\n✅ Classical training completed (fresh run).")

🔷 Starting CLASSICAL model training from scratch (no resume)...

🚀 [Classical] Training — Epoch 1/100


Ep   1/100 [Classical]:   0%|          | 0/2188 [00:00<?, ?it/s]

  Ep  1 | Train=3.1478 | Val=2.8401 | CER=0.8698 | WER=1.0000 | 2.2min


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

  Ep  9 | Train=2.8032 | Val=2.8117 | CER=0.8709 | WER=1.0000 | 2.1min


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Ep  10/100 [Classical]:   0%|          | 0/2188 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7d6509e9a660>
Traceback (most recent call last):
Exception ignored in: Exception ignored in:   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
<function _MultiProcessingDataLoaderIter.__del__ at 0x7d6509e9a660><function _MultiProcessingDataLoaderIter.__del__ at 0x7d6509e9a660>

    self._shutdown_workers()Traceback (most recent call last):

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
      File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    if w.is_alive():    self._shutdown_workers()
self._shutdown_workers() 
Exception ignored in: 
 Exception ignored in:   File "/usr/local/lib/python3.12/dist-packages/t

## Cell 11 — Training Curves & Summary

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Quantum ⚡ vs Classical 🔷 — Training History (Kaggle)', fontsize=14, fontweight='bold')

def plot_compare(ax, q_vals, c_vals, title, ylabel):
    ax.plot(range(1, len(q_vals)+1), q_vals, '#7B2D8B', lw=2, label='Quantum ⚡')
    ax.plot(range(1, len(c_vals)+1), c_vals, '#2196F3', lw=2, label='Classical 🔷', ls='--')
    ax.fill_between(range(1, len(q_vals)+1), q_vals, alpha=0.1, color='#7B2D8B')
    ax.set_title(title, fontweight='bold'); ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel)
    ax.legend(); ax.grid(alpha=0.3)

plot_compare(axes[0], quantum_history['val_loss'], classical_history['val_loss'], 'Val Loss', 'CTC Loss')
plot_compare(axes[1], quantum_history['val_cer'],  classical_history['val_cer'],  'CER ↓', 'CER')
plot_compare(axes[2], [1-v for v in quantum_history['val_wer']],
                      [1-v for v in classical_history['val_wer']],
                      'Plate Accuracy ↑', 'Accuracy')

plt.tight_layout()
curves_path = '/kaggle/working/training_curves.png'
plt.savefig(curves_path, dpi=150, bbox_inches='tight')
plt.show()

# Upload plot to HF
try:
    api.upload_file(path_or_fileobj=curves_path, path_in_repo='results/training_curves.png',
                    repo_id=HF_MODEL_REPO, repo_type='model', token=active_token)
    print('✅ Training curves pushed to HuggingFace.')
except Exception as e:
    print(f'⚠️  Could not push plot: {e}')

# Best epoch summary
for hist, name in [(quantum_history,'Quantum ⚡'), (classical_history,'Classical 🔷')]:
    if hist['val_cer']:
        best = int(np.argmin(hist['val_cer']))
        print(f'  [{name}] Best ep {best+1}: '
              f'CER={hist["val_cer"][best]*100:.1f}%  '
              f'Acc={(1-hist["val_wer"][best])*100:.1f}%')

In [ ]:
# # ⚠️ RUN THIS CELL ONLY ONCE – it permanently deletes classical model files from HF

# from huggingface_hub import HfApi

# # Use the already authenticated API (active_token and api from earlier cells)
# print("🗑️ Deleting classical model files from Hugging Face...")

# files_to_delete = [
#     "classical/latest.pth",
#     "classical/best.pth",
#     "classical/history.json",
# ]

# for file_path in files_to_delete:
#     try:
#         api.delete_file(
#             path_in_repo=file_path,
#             repo_id=HF_MODEL_REPO,
#             repo_type="model",
#             token=active_token,
#         )
#         print(f"  ✅ Deleted {file_path}")
#     except Exception as e:
#         print(f"  ⚠️ Could not delete {file_path} (maybe already gone): {e}")

# print("✅ Classical model artifacts removed from Hugging Face.")